# LBG Classification Model Training

This notebook trains and evaluates LightGBM models for LBG classification. 

## Strategy
- Phase 1: Train on 70%, evaluate on 30% test set
- Phase 2: Train final model on 100% data
- Compare 3 feature sets

In [1]:
# Cell 1: Imports and Configuration

import numpy as np
import pandas as pd
import matplotlib. pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
import lightgbm as lgb
import warnings
import os
import joblib
from datetime import datetime

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Imports completed.")
print(f"LightGBM version: {lgb.__version__}")

Imports completed.
LightGBM version: 4.6.0


In [2]:
# Cell 2: Configuration

# =============================================================================
# CONFIGURATION
# =============================================================================

# --- Paths ---
DATA_PATH = "../data_processed/training_dataset.csv"
MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Random seed for reproducibility ---
RANDOM_STATE = 42

# --- Train/Test split ---
TEST_SIZE = 0.30  # 30% for testing
CV_FOLDS = 5      # 5-fold cross-validation

# --- Feature sets to compare ---
FEATURE_SETS = {
    'full': [
        'mag_u', 'mag_g', 'mag_r', 'mag_i', 'mag_z', 'mag_y',
        'err_u', 'err_g', 'err_r', 'err_i', 'err_z', 'err_y',
        'u_g', 'g_r', 'r_i', 'i_z', 'z_y'
    ],
    'core_flag': [
        'u_g', 'g_r', 'r_i', 'i_z', 'z_y',
        'mag_i',
        'flag_u', 'flag_g', 'flag_r', 'flag_i', 'flag_z', 'flag_y'
    ],
    'core_err': [
        'u_g', 'g_r', 'r_i', 'i_z', 'z_y',
        'mag_i',
        'err_u', 'err_g', 'err_r', 'err_i', 'err_z', 'err_y'
    ]
}

# --- LightGBM parameters ---
LGBM_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves':  31,
    'max_depth': -1,
    'learning_rate': 0.05,
    'n_estimators': 500,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbose': -1,
}

# --- Label column ---
LABEL_COL = 'is_lbg'

print("Configuration loaded.")
print(f"Test size: {TEST_SIZE*100:.0f}%")
print(f"CV folds: {CV_FOLDS}")
print(f"Feature sets: {list(FEATURE_SETS.keys())}")


Configuration loaded.
Test size: 30%
CV folds: 5
Feature sets: ['full', 'core_flag', 'core_err']


In [3]:
# Cell 3: Load and Prepare Data

# Load data
df = pd.read_csv(DATA_PATH)
print(f"Loaded data: {df.shape[0]} samples, {df.shape[1]} columns")

# Check class distribution
print(f"\nClass distribution:")
print(df[LABEL_COL].value_counts())
print(f"Class ratio (LBG/Non-LBG): {df[LABEL_COL].mean():.3f}")

# Check for missing values in features
all_features = set()
for features in FEATURE_SETS.values():
    all_features.update(features)

print(f"\nMissing values in features:")
for col in sorted(all_features):
    if col in df.columns:
        n_missing = df[col].isna().sum()
        if n_missing > 0:
            print(f"  {col}: {n_missing} ({n_missing/len(df)*100:.2f}%)")

Loaded data: 5420 samples, 40 columns

Class distribution:
is_lbg
 1    3023
 0    2377
-1      20
Name: count, dtype: int64
Class ratio (LBG/Non-LBG): 0.554

Missing values in features:
